# 📦 Notebook 04: Structured Outputs (JSON/XML)

**Time:** 25 minutes  
**Goal:** Generate reliable, parseable structured data from LLMs

## Why Structured Outputs?

Natural language is great for humans, but programs need structured data:

**Problem:** LLMs naturally produce text  
**Solution:** Train them to output JSON, XML, or other formats

**Use Cases:**
- Extracting data from documents
- Building APIs that consume LLM outputs
- Creating databases from unstructured text
- Chaining LLM calls together
- Integrating with other systems

## What You'll Learn

- JSON output formatting
- XML output formatting  
- Schema validation with Pydantic
- Error handling strategies
- Parsing and using structured data
- Ensuring consistency

Let's turn messy text into clean data! 🎯

**Prerequisites:** Notebooks 02 and 03 completed

In [4]:
# Setup and Imports
import os
import sys
from pathlib import Path
import time
import json
import xml.etree.ElementTree as ET
from typing import Dict, List, Any, Optional

# Add parent directory to path
notebook_dir = os.getcwd()
parent_dir = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Load environment
from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

# Import our modules
from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import estimate_tokens, estimate_cost, append_to_reflection
from src.config import PATH

# Pydantic for validation
try:
    from pydantic import BaseModel, Field, ValidationError
    print("✓ Pydantic available for validation")
except ImportError:
    print("⚠ Pydantic not installed - validation examples will be limited")
    print("  Install with: pip install pydantic")

print("=" * 60)
print("NOTEBOOK 04: STRUCTURED OUTPUTS")
print("=" * 60)
print()
print(f"Configuration loaded: Path {PATH}")
print()

# Initialize client and tracker
client = LLMClient(path=PATH)
tracker = CostTracker()

print()
print("✓ Ready to learn structured outputs!")
print()

✓ Pydantic available for validation
NOTEBOOK 04: STRUCTURED OUTPUTS

Configuration loaded: Path A

✓ Claude API client initialized
  Default model: claude-sonnet-4-5-20250929
  Available: Opus 4.5, Sonnet 4.5, Haiku 4.5

✓ Ready to learn structured outputs!



---
## 📄 Part 1: JSON Outputs

JSON (JavaScript Object Notation) is the most common structured format for APIs and data exchange.

### Why JSON?

- ✅ Human-readable
- ✅ Easy to parse in any language
- ✅ Supports nested structures
- ✅ Standard for web APIs
- ✅ Native Python support

### 🎯 The Challenge

LLMs want to be helpful and chatty:

**Bad LLM Output:**
```
Sure! Here's the information you requested in JSON format:

{
  "name": "John"
}

Hope that helps!
```

**Good LLM Output:**
```json
{
  "name": "John"
}
```

We need to train them to output ONLY valid JSON with no extra text.

In [5]:
# Basic JSON Output
print("=" * 60)
print("EXPERIMENT 1: Basic JSON Output")
print("=" * 60)
print()

# Bad prompt - no structure specified
bad_prompt = "Extract the person's name and age from this text: 'John is 25 years old'"

print("❌ BAD PROMPT (no structure specified):")
print("-" * 60)
print(bad_prompt)
print()

response = client.generate(
    prompt=bad_prompt,
    temperature=0.0,
    max_tokens=100
)

if "error" not in response:
    print("Response:")
    print(response['content'])
    print()
    print("Problem: This might not be valid JSON!")
    tracker.add_call(response)
print()
print()

# Good prompt - explicit JSON instruction
good_prompt = """Extract the person's name and age from this text: 'John is 25 years old'

Respond with ONLY valid JSON in this exact format, with no additional text:
{
  "name": "string",
  "age": number
}"""

print("✅ GOOD PROMPT (explicit JSON format):")
print("-" * 60)
print(good_prompt)
print()

response = client.generate(
    prompt=good_prompt,
    temperature=0.0,
    max_tokens=100
)

if "error" not in response:
    print("Response:")
    print(response['content'])
    print()
    
    # Try to parse it
    try:
        parsed = json.loads(response['content'])
        print("✓ Successfully parsed as JSON!")
        print(f"✓ Name: {parsed.get('name')}")
        print(f"✓ Age: {parsed.get('age')}")
    except json.JSONDecodeError as e:
        print(f"✗ Failed to parse: {e}")
    
    tracker.add_call(response)

print()

EXPERIMENT 1: Basic JSON Output

❌ BAD PROMPT (no structure specified):
------------------------------------------------------------
Extract the person's name and age from this text: 'John is 25 years old'

Response:
**Name:** John  
**Age:** 25

Problem: This might not be valid JSON!


✅ GOOD PROMPT (explicit JSON format):
------------------------------------------------------------
Extract the person's name and age from this text: 'John is 25 years old'

Respond with ONLY valid JSON in this exact format, with no additional text:
{
  "name": "string",
  "age": number
}

Response:
```json
{
  "name": "John",
  "age": 25
}
```

✗ Failed to parse: Expecting value: line 1 column 1 (char 0)



### 🔑 Key Patterns for JSON Output

**Pattern 1: Explicit Schema**
```
Respond with ONLY valid JSON in this format:
{
  "field1": "type",
  "field2": number
}
```

**Pattern 2: System Prompt**
```
You always respond in valid JSON format with no additional text.
```

**Pattern 3: Examples**
```
Example output:
{"name": "Alice", "age": 30}

Now extract from: "Bob is 25"
```

In [6]:
# Cell 3: System Prompt for JSON
print("=" * 60)
print("EXPERIMENT 2: Using System Prompt for JSON")
print("=" * 60)
print()

# Set up a JSON-only system prompt
json_system = """You are a data extraction assistant.
You ALWAYS respond with ONLY valid JSON, with no additional text before or after.
Never include explanations, apologies, or markdown formatting.
Output pure JSON only."""

user_prompt = """Extract information from this text:
"Sarah Chen is a software engineer at Google. She's 28 years old and lives in San Francisco."

Use this JSON schema:
{
  "name": "string",
  "occupation": "string",
  "company": "string",
  "age": number,
  "city": "string"
}"""

print("System Prompt:")
print("-" * 60)
print(json_system)
print()
print("User Prompt:")
print("-" * 60)
print(user_prompt)
print()

response = client.generate(
    prompt=user_prompt,
    system=json_system,
    temperature=0.0,
    max_tokens=150
)

if "error" not in response:
    print("Response:")
    print(response['content'])
    print()
    
    # Parse and validate
    try:
        data = json.loads(response['content'])
        print("✅ Valid JSON!")
        print()
        print("Parsed data:")
        for key, value in data.items():
            print(f"  {key}: {value}")
    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing failed: {e}")
    
    tracker.add_call(response)

print()

EXPERIMENT 2: Using System Prompt for JSON

System Prompt:
------------------------------------------------------------
You are a data extraction assistant.
You ALWAYS respond with ONLY valid JSON, with no additional text before or after.
Never include explanations, apologies, or markdown formatting.
Output pure JSON only.

User Prompt:
------------------------------------------------------------
Extract information from this text:
"Sarah Chen is a software engineer at Google. She's 28 years old and lives in San Francisco."

Use this JSON schema:
{
  "name": "string",
  "occupation": "string",
  "company": "string",
  "age": number,
  "city": "string"
}

Response:
```json
{
  "name": "Sarah Chen",
  "occupation": "software engineer",
  "company": "Google",
  "age": 28,
  "city": "San Francisco"
}
```

❌ JSON parsing failed: Expecting value: line 1 column 1 (char 0)



---
## 🛡️ Part 2: Schema Validation with Pydantic

Pydantic lets you define schemas and automatically validate data.

**Benefits:**
- Type checking
- Required field validation
- Default values
- Custom validation rules
- Automatic error messages

In [7]:
# Pydantic Schema Validation
print("=" * 60)
print("EXPERIMENT 3: Pydantic Schema Validation")
print("=" * 60)
print()

# Define a schema
class Person(BaseModel):
    name: str = Field(..., description="Full name of the person")
    age: int = Field(..., ge=0, le=150, description="Age in years")
    occupation: str = Field(..., description="Job title or profession")
    city: str = Field(..., description="City of residence")
    email: Optional[str] = Field(None, description="Email address if available")

print("Pydantic Schema:")
print("-" * 60)
print(Person.model_json_schema())
print()

# Create a prompt with the schema
schema_prompt = f"""Extract information from this text:
"Alice Johnson, age 32, works as a data scientist in Seattle. Her email is alice.j@email.com"

Respond with ONLY valid JSON matching this schema:
{json.dumps(Person.model_json_schema(), indent=2)}

Output ONLY the JSON object, no other text."""

response = client.generate(
    prompt=schema_prompt,
    temperature=0.0,
    max_tokens=200
)

if "error" not in response:
    print("LLM Response:")
    print(response['content'])
    print()
    
    # Parse and validate with Pydantic
    try:
        # Clean the response (remove markdown if present)
        cleaned = response['content'].strip()
        if cleaned.startswith('```'):
            # Remove markdown code blocks
            cleaned = '\n'.join(cleaned.split('\n')[1:-1])
        
        data = json.loads(cleaned)
        person = Person(**data)
        
        print("✅ Validation successful!")
        print()
        print("Validated data:")
        print(f"  Name: {person.name}")
        print(f"  Age: {person.age}")
        print(f"  Occupation: {person.occupation}")
        print(f"  City: {person.city}")
        print(f"  Email: {person.email}")
        
    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing error: {e}")
    except ValidationError as e:
        print(f"❌ Validation error:")
        print(e)
    
    tracker.add_call(response)

print()

EXPERIMENT 3: Pydantic Schema Validation

Pydantic Schema:
------------------------------------------------------------
{'properties': {'name': {'description': 'Full name of the person', 'title': 'Name', 'type': 'string'}, 'age': {'description': 'Age in years', 'maximum': 150, 'minimum': 0, 'title': 'Age', 'type': 'integer'}, 'occupation': {'description': 'Job title or profession', 'title': 'Occupation', 'type': 'string'}, 'city': {'description': 'City of residence', 'title': 'City', 'type': 'string'}, 'email': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Email address if available', 'title': 'Email'}}, 'required': ['name', 'age', 'occupation', 'city'], 'title': 'Person', 'type': 'object'}

LLM Response:
```json
{
  "name": "Alice Johnson",
  "age": 32,
  "occupation": "data scientist",
  "city": "Seattle",
  "email": "alice.j@email.com"
}
```

✅ Validation successful!

Validated data:
  Name: Alice Johnson
  Age: 32
  Occupation: data scientist
  

---
## 🔄 Part 3: Handling Complex Structures

Real-world data often has nested objects and arrays.

In [8]:
# Complex Nested JSON
print("=" * 60)
print("EXPERIMENT 4: Complex Nested JSON")
print("=" * 60)
print()

# Define a complex schema
class Product(BaseModel):
    name: str
    price: float
    in_stock: bool

class Order(BaseModel):
    order_id: str
    customer_name: str
    products: List[Product]
    total: float
    shipping_address: Dict[str, str]

print("Complex Schema:")
print("-" * 60)
print(json.dumps(Order.model_json_schema(), indent=2))
print()

complex_prompt = """Extract order information from this text:

"Order #12345 for John Smith. He ordered a laptop for $999 (in stock) and a mouse for $25 (in stock). 
Total: $1024. Ship to: 123 Main St, Apt 4B, Seattle, WA 98101"

Respond with ONLY valid JSON matching this structure:
{
  "order_id": "string",
  "customer_name": "string",
  "products": [
    {
      "name": "string",
      "price": number,
      "in_stock": boolean
    }
  ],
  "total": number,
  "shipping_address": {
    "street": "string",
    "unit": "string",
    "city": "string",
    "state": "string",
    "zip": "string"
  }
}

Output ONLY the JSON, no other text."""

response = client.generate(
    prompt=complex_prompt,
    temperature=0.0,
    max_tokens=300
)

if "error" not in response:
    print("LLM Response:")
    print(response['content'])
    print()
    
    # Parse and validate
    try:
        cleaned = response['content'].strip()
        if cleaned.startswith('```'):
            cleaned = '\n'.join(cleaned.split('\n')[1:-1])
        
        data = json.loads(cleaned)
        order = Order(**data)
        
        print("✅ Valid complex JSON!")
        print()
        print(f"Order ID: {order.order_id}")
        print(f"Customer: {order.customer_name}")
        print(f"Products ({len(order.products)}):")
        for p in order.products:
            print(f"  - {p.name}: ${p.price} (in stock: {p.in_stock})")
        print(f"Total: ${order.total}")
        print(f"Shipping: {order.shipping_address}")
        
    except (json.JSONDecodeError, ValidationError) as e:
        print(f"❌ Error: {e}")
    
    tracker.add_call(response)

print()

EXPERIMENT 4: Complex Nested JSON

Complex Schema:
------------------------------------------------------------
{
  "$defs": {
    "Product": {
      "properties": {
        "name": {
          "title": "Name",
          "type": "string"
        },
        "price": {
          "title": "Price",
          "type": "number"
        },
        "in_stock": {
          "title": "In Stock",
          "type": "boolean"
        }
      },
      "required": [
        "name",
        "price",
        "in_stock"
      ],
      "title": "Product",
      "type": "object"
    }
  },
  "properties": {
    "order_id": {
      "title": "Order Id",
      "type": "string"
    },
    "customer_name": {
      "title": "Customer Name",
      "type": "string"
    },
    "products": {
      "items": {
        "$ref": "#/$defs/Product"
      },
      "title": "Products",
      "type": "array"
    },
    "total": {
      "title": "Total",
      "type": "number"
    },
    "shipping_address": {
      "additionalP

---
## 📋 Part 4: XML Outputs

XML is common in enterprise systems and legacy APIs.

**When to use XML:**
- Working with enterprise systems
- SOAP APIs
- Configuration files
- Document markup

In [9]:
# XML Output Format
print("=" * 60)
print("EXPERIMENT 5: XML Output Format")
print("=" * 60)
print()

xml_prompt = """Extract book information from this text:
"The Great Gatsby by F. Scott Fitzgerald, published in 1925. ISBN: 978-0-7432-7356-5. Genre: Fiction. 180 pages."

Respond with ONLY valid XML in this format:
<book>
  <title>string</title>
  <author>string</author>
  <year>number</year>
  <isbn>string</isbn>
  <genre>string</genre>
  <pages>number</pages>
</book>

Output ONLY the XML, no other text."""

response = client.generate(
    prompt=xml_prompt,
    temperature=0.0,
    max_tokens=200
)

if "error" not in response:
    print("LLM Response:")
    print(response['content'])
    print()
    
    # Parse XML
    try:
        cleaned = response['content'].strip()
        if cleaned.startswith('```'):
            cleaned = '\n'.join(cleaned.split('\n')[1:-1])
        
        root = ET.fromstring(cleaned)
        
        print("✅ Valid XML!")
        print()
        print("Parsed data:")
        for child in root:
            print(f"  {child.tag}: {child.text}")
            
    except ET.ParseError as e:
        print(f"❌ XML parsing error: {e}")
    
    tracker.add_call(response)

print()

EXPERIMENT 5: XML Output Format

LLM Response:
```xml
<book>
  <title>The Great Gatsby</title>
  <author>F. Scott Fitzgerald</author>
  <year>1925</year>
  <isbn>978-0-7432-7356-5</isbn>
  <genre>Fiction</genre>
  <pages>180</pages>
</book>
```

✅ Valid XML!

Parsed data:
  title: The Great Gatsby
  author: F. Scott Fitzgerald
  year: 1925
  isbn: 978-0-7432-7356-5
  genre: Fiction
  pages: 180



---
## 🛠️ Part 5: Error Handling Strategies

LLMs sometimes mess up formatting. Here's how to handle it.

In [10]:
# Robust Parsing Function
print("=" * 60)
print("EXPERIMENT 6: Robust Error Handling")
print("=" * 60)
print()

def parse_json_response(response_text: str, max_attempts: int = 3) -> Optional[Dict]:
    """
    Robustly parse JSON from LLM response.
    
    Handles common issues:
    - Markdown code blocks
    - Extra text before/after JSON
    - Trailing commas
    - Single quotes instead of double quotes
    """
    import re
    
    text = response_text.strip()
    
    # Attempt 1: Direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    
    # Attempt 2: Remove markdown code blocks
    if '```' in text:
        text = re.sub(r'```(?:json)?\n?', '', text)
        text = text.strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass
    
    # Attempt 3: Extract JSON object
    json_match = re.search(r'\{.*\}', text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    
    # Attempt 4: Try to fix common issues
    text = text.replace("'", '"')  # Single to double quotes
    text = re.sub(r',(\s*[}\]])', r'\1', text)  # Remove trailing commas
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    
    return None

# Test with problematic outputs
test_outputs = [
    '{"name": "Alice", "age": 30}',  # Clean
    '```json\n{"name": "Bob", "age": 25}\n```',  # Markdown
    'Sure! Here is the data: {"name": "Carol", "age": 35}',  # Extra text
    "{'name': 'Dave', 'age': 40}",  # Single quotes
    '{"name": "Eve", "age": 45,}',  # Trailing comma
]

print("Testing robust parser on various inputs:")
print()

for i, output in enumerate(test_outputs, 1):
    print(f"Test {i}:")
    print(f"  Input: {output[:50]}...")
    result = parse_json_response(output)
    if result:
        print(f"  ✅ Parsed: {result}")
    else:
        print(f"  ❌ Failed to parse")
    print()

print("💡 Use robust parsing in production to handle LLM variability!")
print()

EXPERIMENT 6: Robust Error Handling

Testing robust parser on various inputs:

Test 1:
  Input: {"name": "Alice", "age": 30}...
  ✅ Parsed: {'name': 'Alice', 'age': 30}

Test 2:
  Input: ```json
{"name": "Bob", "age": 25}
```...
  ✅ Parsed: {'name': 'Bob', 'age': 25}

Test 3:
  Input: Sure! Here is the data: {"name": "Carol", "age": 3...
  ✅ Parsed: {'name': 'Carol', 'age': 35}

Test 4:
  Input: {'name': 'Dave', 'age': 40}...
  ✅ Parsed: {'name': 'Dave', 'age': 40}

Test 5:
  Input: {"name": "Eve", "age": 45,}...
  ✅ Parsed: {'name': 'Eve', 'age': 45}

💡 Use robust parsing in production to handle LLM variability!



---
## 🎯 Your Turn: Practice Tasks

Time to practice generating and parsing structured outputs!

### 📝 Task 1: Build a JSON Extractor

**Goal:** Extract structured data from unstructured text.

**Scenario:** You're building a system to parse customer reviews.

In [11]:
# TODO - Task 1: JSON Extractor
print("=" * 60)
print("TASK 1: Build a JSON Extractor")
print("=" * 60)
print()

# Define schema for product review
class ProductReview(BaseModel):
    product_name: str = Field(..., description="Name of the product")
    rating: int = Field(..., ge=1, le=5, description="Rating from 1-5 stars")
    reviewer_name: str = Field(..., description="Name of the reviewer")
    pros: List[str] = Field(..., description="List of positive aspects")
    cons: List[str] = Field(..., description="List of negative aspects")
    would_recommend: bool = Field(..., description="Whether they recommend it")

print("Schema:")
print(json.dumps(ProductReview.model_json_schema(), indent=2))
print()

# ============================================================================
# TODO: Create a sample review text
# ============================================================================

sample_review = """
# YOUR SAMPLE REVIEW HERE

Example:
"I bought the UltraBook Pro last month and I'm really impressed! The battery life 
is amazing - easily lasts 12 hours. The display is crisp and bright. However, it's 
a bit heavy to carry around, and the keyboard could be more comfortable. Overall, 
I'd definitely recommend it to anyone looking for a powerful laptop. 
- Sarah M. (4 stars)"
"""

# ============================================================================
# TODO: Build your extraction prompt
# ============================================================================

extraction_prompt = f"""
# YOUR PROMPT HERE

Remember to:
- Include the schema
- Ask for ONLY JSON output
- Be clear about the format
"""

print("Sample Review:")
print("-" * 60)
print(sample_review)
print()

print("Your Extraction Prompt:")
print("-" * 60)
print(extraction_prompt)
print()

response = client.generate(
    prompt=extraction_prompt,
    temperature=0.0,
    max_tokens=300
)

if "error" not in response:
    print("LLM Response:")
    print("=" * 60)
    print(response['content'])
    print("=" * 60)
    print()
    
    # Parse and validate
    parsed_data = parse_json_response(response['content'])
    
    if parsed_data:
        try:
            review = ProductReview(**parsed_data)
            
            print("✅ Successfully extracted and validated!")
            print()
            print(f"Product: {review.product_name}")
            print(f"Rating: {'⭐' * review.rating}")
            print(f"Reviewer: {review.reviewer_name}")
            print(f"Pros: {', '.join(review.pros)}")
            print(f"Cons: {', '.join(review.cons)}")
            print(f"Recommends: {'Yes' if review.would_recommend else 'No'}")
            
        except ValidationError as e:
            print(f"❌ Validation failed: {e}")
    else:
        print("❌ Failed to parse JSON")
    
    tracker.add_call(response)
    
    # ========================================================================
    # TODO: Reflection
    # ========================================================================
    
    print()
    print("=" * 60)
    print("REFLECTION")
    print("=" * 60)
    print()
    
    reflection = """
### Extraction Quality

**Did it correctly extract all fields?** [Yes/No]

[What worked well? What was missed?]

### Prompt Effectiveness

**How clear was your prompt?** [Rate 1-5]: ___/5

[What would you change to improve it?]

### Real-World Application

**Where would you use this review extractor?**

[Describe a practical use case]

### Challenges Encountered

[What was difficult about getting structured output?]

[How did you solve it?]
"""
    
    print(reflection)
    
    append_to_reflection(
        notebook="04",
        section_title="Task 1 - JSON Extractor",
        reflection_content=reflection,
        output_dir=os.path.join(parent_dir, 'outputs')
    )
    
    print()
    print("💾 Reflection saved to outputs/homework_reflection.md")

else:
    print(f"❌ Error: {response['error']}")

print()

TASK 1: Build a JSON Extractor

Schema:
{
  "properties": {
    "product_name": {
      "description": "Name of the product",
      "title": "Product Name",
      "type": "string"
    },
    "rating": {
      "description": "Rating from 1-5 stars",
      "maximum": 5,
      "minimum": 1,
      "title": "Rating",
      "type": "integer"
    },
    "reviewer_name": {
      "description": "Name of the reviewer",
      "title": "Reviewer Name",
      "type": "string"
    },
    "pros": {
      "description": "List of positive aspects",
      "items": {
        "type": "string"
      },
      "title": "Pros",
      "type": "array"
    },
    "cons": {
      "description": "List of negative aspects",
      "items": {
        "type": "string"
      },
      "title": "Cons",
      "type": "array"
    },
    "would_recommend": {
      "description": "Whether they recommend it",
      "title": "Would Recommend",
      "type": "boolean"
    }
  },
  "required": [
    "product_name",
    "rating",

### 📝 Task 2: Multi-Record Extraction

**Goal:** Extract multiple records from a single text.

**Scenario:** Parse a list of items from text.

In [12]:
# TODO - Task 2: Multi-Record Extraction
print("=" * 60)
print("TASK 2: Multi-Record Extraction")
print("=" * 60)
print()

# Schema for multiple items
class MenuItem(BaseModel):
    name: str
    category: str
    price: float
    vegetarian: bool
    spicy_level: Optional[int] = Field(None, ge=0, le=3)

class Menu(BaseModel):
    restaurant_name: str
    items: List[MenuItem]

# ============================================================================
# TODO: Create sample menu text with multiple items
# ============================================================================

menu_text = """
# YOUR MENU TEXT HERE

Example:
"Welcome to Tasty Bites! Our menu includes:
- Margherita Pizza (Pizza, $12.99, vegetarian)
- Spicy Chicken Wings (Appetizer, $8.99, spicy level 3)
- Caesar Salad (Salad, $7.99, vegetarian)
- Beef Burger (Main, $14.99)
- Pad Thai (Noodles, $13.99, spicy level 2)"
"""

# ============================================================================
# TODO: Build extraction prompt for multiple items
# ============================================================================

multi_prompt = f"""
# YOUR PROMPT HERE

Extract ALL menu items from the text below.

Schema:
{json.dumps(Menu.model_json_schema(), indent=2)}

Text:
{menu_text}

Respond with ONLY valid JSON, no other text.
"""

print("Menu Text:")
print("-" * 60)
print(menu_text)
print()

response = client.generate(
    prompt=multi_prompt,
    temperature=0.0,
    max_tokens=500
)

if "error" not in response:
    print("LLM Response:")
    print("=" * 60)
    print(response['content'])
    print("=" * 60)
    print()
    
    parsed_data = parse_json_response(response['content'])
    
    if parsed_data:
        try:
            menu = Menu(**parsed_data)
            
            print(f"✅ Extracted {len(menu.items)} items from {menu.restaurant_name}!")
            print()
            
            for i, item in enumerate(menu.items, 1):
                spicy = f" {'🌶️' * item.spicy_level}" if item.spicy_level else ""
                veggie = " 🥬" if item.vegetarian else ""
                print(f"{i}. {item.name}{veggie}{spicy}")
                print(f"   {item.category} - ${item.price:.2f}")
            
        except ValidationError as e:
            print(f"❌ Validation failed: {e}")
    else:
        print("❌ Failed to parse JSON")
    
    tracker.add_call(response)
    
    # ========================================================================
    # TODO: Reflection
    # ========================================================================
    
    print()
    print("=" * 60)
    print("REFLECTION")
    print("=" * 60)
    print()
    
    reflection = """
### Multi-Record Extraction

**Number of items in text:** [X]

**Number correctly extracted:** [X]

**Accuracy:** [X/X = XX%]

### Challenges with Arrays

[What was harder about extracting multiple items vs single items?]

### Data Quality

[Were all fields correctly identified for each item?]

[Any patterns in what was missed or wrong?]

### Scaling Considerations

**How would this work with 100 items?**

[Discuss token limits, cost, reliability]

### Production Readiness

[Would you trust this for production use? Why or why not?]

[What safeguards would you add?]
"""
    
    print(reflection)
    
    append_to_reflection(
        notebook="04",
        section_title="Task 2 - Multi-Record Extraction",
        reflection_content=reflection,
        output_dir=os.path.join(parent_dir, 'outputs')
    )
    
    print()
    print("💾 Reflection saved to outputs/homework_reflection.md")

else:
    print(f"❌ Error: {response['error']}")

print()

TASK 2: Multi-Record Extraction

Menu Text:
------------------------------------------------------------

# YOUR MENU TEXT HERE

Example:
"Welcome to Tasty Bites! Our menu includes:
- Margherita Pizza (Pizza, $12.99, vegetarian)
- Spicy Chicken Wings (Appetizer, $8.99, spicy level 3)
- Caesar Salad (Salad, $7.99, vegetarian)
- Beef Burger (Main, $14.99)
- Pad Thai (Noodles, $13.99, spicy level 2)"


LLM Response:
```json
{
  "restaurant_name": "Tasty Bites",
  "items": [
    {
      "name": "Margherita Pizza",
      "category": "Pizza",
      "price": 12.99,
      "vegetarian": true,
      "spicy_level": null
    },
    {
      "name": "Spicy Chicken Wings",
      "category": "Appetizer",
      "price": 8.99,
      "vegetarian": false,
      "spicy_level": 3
    },
    {
      "name": "Caesar Salad",
      "category": "Salad",
      "price": 7.99,
      "vegetarian": true,
      "spicy_level": null
    },
    {
      "name": "Beef Burger",
      "category": "Main",
      "price": 14.99

### 📝 Task 3: Error Recovery System

**Goal:** Build a system that retries with better prompts when parsing fails.

**Scenario:** Create a robust extraction pipeline.

In [13]:
# TODO - Task 3: Error Recovery
print("=" * 60)
print("TASK 3: Build an Error Recovery System")
print("=" * 60)
print()

def extract_with_retry(
    text: str,
    schema: BaseModel,
    max_retries: int = 3,
    client: LLMClient = client,
    tracker: CostTracker = tracker
) -> Optional[BaseModel]:
    """
    Attempt to extract structured data with retry logic.
    
    Args:
        text: Text to extract from
        schema: Pydantic model class
        max_retries: Maximum retry attempts
        client: LLM client
        tracker: Cost tracker
    
    Returns:
        Validated schema instance or None
    """
    
    for attempt in range(max_retries):
        print(f"Attempt {attempt + 1}/{max_retries}")
        print("-" * 60)
        
        # Build prompt with increasing specificity
        if attempt == 0:
            # First attempt: basic prompt
            prompt = f"""Extract data from this text in JSON format:

{text}

Use this schema:
{json.dumps(schema.model_json_schema(), indent=2)}

Respond with ONLY valid JSON, no other text."""
            
        elif attempt == 1:
            # Second attempt: more explicit
            prompt = f"""You MUST respond with ONLY valid JSON. No explanations, no markdown, no extra text.

Extract data from:
{text}

Required JSON schema:
{json.dumps(schema.model_json_schema(), indent=2)}

Output format: Pure JSON object only."""
            
        else:
            # Final attempt: example-based
            prompt = f"""CRITICAL: Output ONLY raw JSON. Example of correct output:
{{"field1": "value1", "field2": 123}}

Now extract from:
{text}

Schema:
{json.dumps(schema.model_json_schema(), indent=2)}

Output ONLY the JSON object:"""
        
        response = client.generate(
            prompt=prompt,
            temperature=0.0,
            max_tokens=400
        )
        
        if "error" in response:
            print(f"❌ API error: {response['error']}")
            continue
        
        tracker.add_call(response)
        
        # Try to parse
        parsed = parse_json_response(response['content'])
        
        if parsed:
            try:
                instance = schema(**parsed)
                print(f"✅ Success on attempt {attempt + 1}!")
                return instance
            except ValidationError as e:
                print(f"⚠ Validation failed: {str(e)[:100]}...")
        else:
            print("⚠ JSON parsing failed")
        
        print()
    
    print(f"❌ Failed after {max_retries} attempts")
    return None

# ============================================================================
# TODO: Test the retry system with a difficult extraction
# ============================================================================

# Define a schema
class Event(BaseModel):
    title: str
    date: str
    location: str
    attendees: int
    virtual: bool

# Create challenging text
difficult_text = """
Join us for the Annual Tech Conference on March 15th at the Convention Center 
downtown! Expected attendance: around 500 people. This is an in-person event.
"""

print("Testing retry system...")
print()
print("Text to extract from:")
print(difficult_text)
print()

result = extract_with_retry(
    text=difficult_text,
    schema=Event,
    max_retries=3
)

if result:
    print()
    print("=" * 60)
    print("FINAL RESULT")
    print("=" * 60)
    print(f"Title: {result.title}")
    print(f"Date: {result.date}")
    print(f"Location: {result.location}")
    print(f"Attendees: {result.attendees}")
    print(f"Virtual: {result.virtual}")
else:
    print()
    print("=" * 60)
    print("Extraction failed completely")
    print("=" * 60)

print()

# ========================================================================
# TODO: Reflection
# ========================================================================

print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = """
### Retry System Performance

**Succeeded on attempt:** [1/2/3/Failed]

**What changed between attempts?**

[Describe how the prompt evolved]

### Error Recovery Strategies

**Most effective strategy:** [Basic/Explicit/Example-based]

[Why do you think this worked best?]

### Production Considerations

**Is 3 retries enough?**

no, it's not enough.

**What's the cost of retries?**

[Calculate: X attempts × Y tokens × $Z per token]

**When would you give up?**

[Define failure criteria]

### Alternative Approaches

**Besides retrying, what else could you do?**

1. [Alternative 1]
2. [Alternative 2]
3. [Alternative 3]

### System Design

**How would you integrate this into a production pipeline?**

[Describe the full flow from input to validated output]
"""

print(reflection)

append_to_reflection(
    notebook="04",
    section_title="Task 3 - Error Recovery System",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 3: Build an Error Recovery System

Testing retry system...

Text to extract from:

Join us for the Annual Tech Conference on March 15th at the Convention Center 
downtown! Expected attendance: around 500 people. This is an in-person event.


Attempt 1/3
------------------------------------------------------------
✅ Success on attempt 1!

FINAL RESULT
Title: Annual Tech Conference
Date: March 15th
Location: Convention Center downtown
Attendees: 500
Virtual: False

REFLECTION


### Retry System Performance

**Succeeded on attempt:** [1/2/3/Failed]

**What changed between attempts?**

[Describe how the prompt evolved]

### Error Recovery Strategies

**Most effective strategy:** [Basic/Explicit/Example-based]

[Why do you think this worked best?]

### Production Considerations

**Is 3 retries enough?**

no, it's not enough.

**What's the cost of retries?**

[Calculate: X attempts × Y tokens × $Z per token]

**When would you give up?**

[Define failure criteria]

### Alternative Approa

---
## 📊 Best Practices Summary

In [14]:
# Best Practices for Structured Outputs
print("=" * 60)
print("BEST PRACTICES FOR STRUCTURED OUTPUTS")
print("=" * 60)
print()

best_practices = {
    "Prompting": [
        "✓ Be explicit: 'Respond with ONLY valid JSON'",
        "✓ Provide schema or example",
        "✓ Use temperature=0.0 for consistency",
        "✓ Specify 'no additional text'",
        "✓ Consider system prompt for repeated tasks",
        "✗ Don't assume the LLM knows your schema",
        "✗ Don't leave format ambiguous"
    ],
    
    "Parsing": [
        "✓ Use robust parsing (handle markdown, extra text)",
        "✓ Implement retry logic",
        "✓ Validate with Pydantic or similar",
        "✓ Log parsing failures for debugging",
        "✓ Handle edge cases gracefully",
        "✗ Don't assume perfect JSON output",
        "✗ Don't fail silently"
    ],
    
    "Schema Design": [
        "✓ Keep schemas simple when possible",
        "✓ Use descriptive field names",
        "✓ Include field descriptions",
        "✓ Set validation constraints (min/max, regex)",
        "✓ Make optional fields explicit",
        "✗ Don't over-complicate nested structures",
        "✗ Don't use ambiguous field names"
    ],
    
    "Error Handling": [
        "✓ Implement retry with different prompts",
        "✓ Validate before using data",
        "✓ Log all failures for analysis",
        "✓ Have fallback strategies",
        "✓ Monitor success rates",
        "✗ Don't silently accept bad data",
        "✗ Don't retry indefinitely"
    ],
    
    "Production": [
        "✓ Test with diverse inputs",
        "✓ Monitor extraction accuracy",
        "✓ Cache successful prompts",
        "✓ Version your schemas",
        "✓ Document expected formats",
        "✗ Don't trust 100% accuracy",
        "✗ Don't skip validation"
    ]
}

for category, practices in best_practices.items():
    print(f"🎯 {category}")
    print("-" * 60)
    for practice in practices:
        print(f"  {practice}")
    print()

print()
print("=" * 60)
print("GOLDEN RULES")
print("=" * 60)
print()
print("1. Always validate - never trust LLM output blindly")
print("2. Temperature 0.0 for consistency")
print("3. Be explicit about format in every prompt")
print("4. Build robust parsers that handle variations")
print("5. Monitor and log failures for continuous improvement")
print()

BEST PRACTICES FOR STRUCTURED OUTPUTS

🎯 Prompting
------------------------------------------------------------
  ✓ Be explicit: 'Respond with ONLY valid JSON'
  ✓ Provide schema or example
  ✓ Use temperature=0.0 for consistency
  ✓ Specify 'no additional text'
  ✓ Consider system prompt for repeated tasks
  ✗ Don't assume the LLM knows your schema
  ✗ Don't leave format ambiguous

🎯 Parsing
------------------------------------------------------------
  ✓ Use robust parsing (handle markdown, extra text)
  ✓ Implement retry logic
  ✓ Validate with Pydantic or similar
  ✓ Log parsing failures for debugging
  ✓ Handle edge cases gracefully
  ✗ Don't assume perfect JSON output
  ✗ Don't fail silently

🎯 Schema Design
------------------------------------------------------------
  ✓ Keep schemas simple when possible
  ✓ Use descriptive field names
  ✓ Include field descriptions
  ✓ Set validation constraints (min/max, regex)
  ✓ Make optional fields explicit
  ✗ Don't over-complicate nested

---
## ✅ Notebook 04 Complete!

### Summary

You've mastered structured outputs! You now know:
- ✅ JSON and XML generation
- ✅ Schema validation with Pydantic
- ✅ Robust parsing techniques
- ✅ Error handling and retry logic
- ✅ Multi-record extraction
- ✅ Production best practices

In [ ]:
# Final Reflection
print("=" * 60)
print("OVERALL NOTEBOOK REFLECTION")
print("=" * 60)
print()

# ============================================================================
# TODO: Final reflection on structured outputs
# ============================================================================

reflection = """
### 1. What was the hardest part of getting reliable JSON/XML?

[Your answer]

### 2. How confident are you in building production data extractors? (1-5)

**Confidence:** ___/5

[What would increase your confidence?]

### 3. When would you use JSON vs XML?

[Your criteria for choosing]

### 4. What's your strategy for handling extraction failures?

[Describe your approach]

### 5. How will you use structured outputs in your projects?

[Real-world application ideas]

### 6. Biggest takeaway from this notebook?

[Your key learning]
"""

print(reflection)

# Save reflection
append_to_reflection(
    notebook="04",
    section_title="Overall Reflection",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")

# Show costs
print()
print("=" * 60)
print("YOUR COSTS THIS NOTEBOOK")
print("=" * 60)
print()
tracker.report()

print()
print("=" * 60)
print("✅ NOTEBOOK 04 COMPLETE!")
print("=" * 60)
print()
print("Progress: [██████████████░░░░░░] 50% Complete")
print()
print("✓ Notebook 00: Setup Verification")
print("✓ Notebook 01: Environment Setup")
print("✓ Notebook 02: LLM Basics")
print("✓ Notebook 03: CO-STAR Framework")
print("✓ Notebook 04: Structured Outputs ← YOU ARE HERE")
print("○ Notebook 05: Chain of Thought")
print("○ Notebook 06: Model Comparison")
print("○ Notebook 07: MCP Introduction")
print("○ Notebook 08: Project Kickoff")
print()
print("Next: notebooks/05_chain_of_thought.ipynb")
print()

OVERALL NOTEBOOK REFLECTION


### 1. What was the hardest part of getting reliable JSON/XML?

[Your answer]

### 2. How confident are you in building production data extractors? (1-5)

**Confidence:** ___/5

[What would increase your confidence?]

### 3. When would you use JSON vs XML?

[Your criteria for choosing]

### 4. What's your strategy for handling extraction failures?

[Describe your approach]

### 5. How will you use structured outputs in your projects?

[Real-world application ideas]

### 6. Biggest takeaway from this notebook?

[Your key learning]


💾 Reflection saved to outputs/homework_reflection.md

YOUR COSTS THIS NOTEBOOK

💰 API COST REPORT
Total API calls: 14
Total input tokens: 2,505
Total output tokens: 1,281
Total cost: $0.0267

Recent calls:
  1. [22:33:35] sonnet - 222in/171out - $0.0032
  2. [22:34:09] sonnet - 142in/87out - $0.0017
  3. [22:35:12] sonnet - 38in/300out - $0.0046
  4. [22:38:07] sonnet - 550in/306out - $0.0062
  5. [22:38:36] sonnet - 264in/57out